In [26]:
import numpy as np
print('alert:1')

alert:1


In [27]:
import torch
import torch.nn.functional as nn
import torch.nn as nn
import torch.nn.functional as f
print('Alert 1')

Alert 1


In [28]:
name = open('names.txt','r').read().splitlines()

In [29]:
n = len(name)

In [30]:
char = sorted(list(set(''.join(name))))
stoi = {s:i+1 for i,s in enumerate(char)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [31]:
test ,train= [],[]
for w in name:
    chs = ['.','.'] + list(w) + ['.']
    for ch1,ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        test.append(ix1)
        train.append(ix2)
        
xs = torch.tensor(test,dtype=torch.int64)
ys = torch.tensor(train)
xs.dtype

torch.int64

In [32]:
#data setspliting

n_train = int(n * 0.7)
n_dev = int(n * 0.2)
n_test = int(n_train-n_dev)

In [33]:
xs_train = xs[:n_train]
ys_train = ys[:n_train]


xs_dev = xs[n_train:n_train + n_dev]
ys_dev = ys[n_train:n_train + n_dev]

xs_test = xs[n_train + n_dev:]
ys_test = ys[n_train + n_dev:]


In [34]:
xenc = f.one_hot(xs_train, num_classes=27).float()


In [35]:
for epoch in range(100):   
    model = nn.Linear(27, 27)
    logits = model(xenc)
    loss = f.cross_entropy(logits,ys_train)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 10 == 0:
        print(loss.item())
    

3.3600287437438965
3.3022751808166504
3.3527464866638184
3.269549608230591
3.287642478942871
3.2987594604492188
3.287196636199951
3.324720621109009
3.305072069168091
3.252007484436035


In [36]:
loss.item()

3.3008670806884766

In [37]:
# validation
with torch.no_grad():
    xenc = f.one_hot(xs_dev, num_classes=27).float()
    logits = model(xenc)
    dev_loss = f.cross_entropy(logits,ys_dev)
    print('Dev loss :',dev_loss.item())

Dev loss : 3.2949304580688477


In [38]:
# for test
with torch.no_grad():
    xenc = f.one_hot(xs_test, num_classes=27).float()
    logits = model(xenc)
    train_loss = f.cross_entropy(logits,ys_test)
    print('train loss: ',train_loss.item())

train loss:  3.2908966541290283


In [39]:
# next quetion


In [42]:
alphas = [0,0.001,0.01,0.1,0.5,1]
out = []
for alpha in alphas:
    g = torch.Generator().manual_seed(2147483647)
    w = torch.randn((27,27),generator = g ,requires_grad = True)

    for k in range(200):
        logits = w[xs_train]
        loss = f.cross_entropy(logits,ys_train)+ alpha*(w**2).mean()
        
        w.grad =None
        loss.backward()

        w.data += -50 * w.grad

    #Evalute on training set
    train_logits = w[xs_train]
    train_loss = f.cross_entropy(train_logits,ys_train)
    #Evalute on devlopment
    dev_logits = w[xs_dev]
    dev_loss = f.cross_entropy(dev_logits,ys_dev)
    out.append((alpha,train_loss.item(),dev_loss.item()))
    print(f'Alpha = {alpha:<5}',
          f'Train_loss = {train_loss.item():.4f}',
          f'Dev_loss = {dev_loss.item():.4f}')
          
    

        

Alpha = 0     Train_loss = 2.4065 Dev_loss = 2.4388
Alpha = 0.001 Train_loss = 2.4069 Dev_loss = 2.4391
Alpha = 0.01  Train_loss = 2.4112 Dev_loss = 2.4419
Alpha = 0.1   Train_loss = 2.4637 Dev_loss = 2.4878
Alpha = 0.5   Train_loss = 2.6307 Dev_loss = 2.6486
Alpha = 1     Train_loss = 2.7751 Dev_loss = 2.7908
